# Florence ? Kaggle T4 ?2

Enable Internet and select GPU T4 ?2. Each GPU has one independent FP16 model and processes **BATCH_SIZE=8 pages** at a time. Start with MODE="pilot"; set MODE="full" for all 9,000 pages, optionally using FULL_SLICE. Batch size 8 is a starting point, not a measured optimum: compare pages/s on your runtime before trying 16 or 32. More VRAM usage alone does not imply more speed.

**Timing:** BATCH_SIZE=1 measures single-page latency. BATCH_SIZE>1 stores batch elapsed time / actual batch size in `pages.seconds` (amortized seconds/page, not individual latency). `batch_timings` records actual batch sizes and elapsed times, including smaller batches after CUDA OOM. Loading images, preprocessing, transfer, generation and decoding are timed; warm-up, scoring and SQLite writes are excluded. Batch timing results must be identified separately in the report. Image size, OCR prompt, greedy decoding and 1,024-token cap remain fixed.

Progress is streamed immediately with a global heartbeat every PROGRESS_SECONDS seconds: completion, errors, pages/s and ETA. Worker messages include current batch size and peak allocated/reserved VRAM. ETA uses new pages processed in this session and is provisional, especially on the ten-page pilot. Batches are kept within each region.

Successful predictions are committed after each batch. Rerunning skips successful rows and retries errors for the **same configuration**. This optimized reader creates new config hashes; old pilot checkpoints remain separate. Save both worker SQLite files with Kaggle notebook outputs; restore the latest copies to `/kaggle/working/oprb_results` before resuming. Stop an existing run before starting another.

In [ ]:
from pathlib import Path
import sys, os, subprocess, tempfile, json
MODE = "pilot"
FULL_SLICE = slice(None)
if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Open this notebook on Kaggle.")
OUTPUT = Path("/kaggle/working/oprb_results")
GPU_WORKERS = 2
BATCH_SIZE = 8  # per GPU; use 1 for the report's original single-page latency protocol
CPU_THREADS = 2  # per GPU worker
IMAGE_THREADS = 2  # parallel image decoding per worker
PROGRESS_SECONDS = 15
assert BATCH_SIZE >= 1 and CPU_THREADS >= 1 and IMAGE_THREADS >= 1
assert PROGRESS_SECONDS > 0
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() < GPU_WORKERS:
    raise RuntimeError("Select GPU T4 x2 in Kaggle, or set GPU_WORKERS=1 on a single GPU.")
import venv
from jupyter_client import KernelManager
from IPython.core.magic import register_cell_magic
runtime = Path(tempfile.gettempdir()) / ('oprb-private-florence' + "-" + sys.implementation.cache_tag)
python = runtime / "bin/python"
if not python.exists():
    venv.create(runtime, with_pip=False)
pip = [sys.executable, "-I", "-m", "pip", "--python", str(python)]
subprocess.run(pip + ["install", "--index-url", "https://download.pytorch.org/whl/cu130", "torch==2.11.0+cu130", "torchvision==0.26.0+cu130"], check=True)
requirements = ['jiwer==4.0.0', 'pandas>=2,<4', 'ipykernel>=6,<8', 'pillow', 'torch==2.11.0+cu130', 'torchvision==0.26.0+cu130', 'transformers==5.23.0', 'accelerate==1.16.0', 'huggingface-hub==2.4.0']
subprocess.run(pip + ["install", *requirements], check=True)
subprocess.run(pip + ["check"], check=True)
if "manager" in globals() and manager.has_kernel:
    client.stop_channels()
    manager.shutdown_kernel(now=True)
manager = KernelManager(kernel_name="python3")
manager.kernel_spec.argv = [str(python), "-I", "-m", "ipykernel_launcher", "-f", "{connection_file}"]
manager.start_kernel(env={k:v for k,v in os.environ.items() if k not in {"PYTHONPATH","PYTHONHOME"}})
client = manager.blocking_client()
client.start_channels()
client.wait_for_ready(timeout=120)

def show_output(message):
    from IPython.display import display
    content = message["content"]
    if message["msg_type"] == "stream":
        print(content["text"], end="", flush=True)
    elif message["msg_type"] in {"display_data", "execute_result"}:
        display(content["data"], raw=True)
    elif message["msg_type"] == "error":
        print("\n".join(content["traceback"]), flush=True)

@register_cell_magic
def benchmark(line, cell):
    try:
        reply = client.execute_interactive(cell, timeout=None, output_hook=show_output)
    except KeyboardInterrupt:
        manager.interrupt_kernel()
        raise
    if reply["content"]["status"] != "ok":
        raise RuntimeError(reply["content"].get("evalue", "Private kernel failed"))

benchmark("", f"from pathlib import Path\nimport os, sys, subprocess, tempfile, json\n"
          f"assert sys.prefix == {str(runtime)!r}\nMODE={MODE!r}\nFULL_SLICE={FULL_SLICE!r}\n"
          f"OUTPUT=Path({str(OUTPUT)!r})\nGPU_WORKERS={GPU_WORKERS}\n"
          f"BATCH_SIZE={BATCH_SIZE}\nCPU_THREADS={CPU_THREADS}\nIMAGE_THREADS={IMAGE_THREADS}\n"
          f"PROGRESS_SECONDS={PROGRESS_SECONDS}")

In [ ]:
%%benchmark
from transformers import AutoProcessor, Florence2ForConditionalGeneration
MODEL = 'florence'
IS_GPU = True
model_python = sys.executable
OUTPUT.mkdir(parents=True, exist_ok=True)
assert MODE in {"pilot","full"}
import torch
if not torch.cuda.is_available() or not 1 <= GPU_WORKERS <= torch.cuda.device_count():
    raise RuntimeError("Select Kaggle T4 x2 or adjust GPU_WORKERS for a single GPU.")
CACHE = Path(tempfile.gettempdir()) / "oprb_test"
import hashlib, json, random, shutil, sqlite3, tarfile, time, unicodedata
from collections import Counter
from importlib.metadata import version
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from PIL import Image
from jiwer import cer, wer
import pandas as pd

REPO = "kpurkayastha/OPRB"
DATA_REV = 'c7ab9fdf2bf33f297bba62e4842d3e5094d11ec2'
ROOT = CACHE / DATA_REV
COUNTS = {"text": 6000, "nontext": 3000}
pages = []
for region, count in COUNTS.items():
    for kind, suffix in [("img", ".jpg"), ("gt", ".txt")]:
        target = ROOT / region / kind
        target.mkdir(parents=True, exist_ok=True)
        if not (target / ".complete").is_file():
            archive = hf_hub_download(REPO, f"test/{region}/{kind}.tar.gz", repo_type="dataset", revision=DATA_REV)
            seen = set()
            with tarfile.open(archive, "r:gz") as tar:
                for member in tar:
                    name = Path(member.name).name
                    if not member.isfile() or Path(name).suffix.lower() != suffix:
                        continue
                    if name in seen:
                        raise ValueError(f"Repeated basename: {name}")
                    seen.add(name)
                    with tar.extractfile(member) as src, (target / name).open("wb") as dest:
                        shutil.copyfileobj(src, dest)
            assert len(seen) == count, (region, kind, len(seen))
            (target / ".complete").touch()
    imgs = {p.stem for p in (ROOT / region / "img").glob("*.jpg")}
    refs = {p.stem for p in (ROOT / region / "gt").glob("*.txt")}
    assert imgs == refs and len(imgs) == count, (region, len(imgs), len(refs))
    pages.extend((region, stem) for stem in sorted(imgs))
selected = pages[FULL_SLICE] if MODE == "full" else [
    key for region in COUNTS
    for key in random.Random(42).sample([p for p in pages if p[0] == region], 5)
]
print(DATA_REV, Counter(r for r, _ in selected))

In [ ]:
%%benchmark
def reference(region, stem):
    tokens = []
    path = ROOT / region / "gt" / (stem + ".txt")
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        if not line.strip():
            continue
        fields = line.split("\t")
        if len(fields) not in (10, 11):
            raise ValueError(f"Malformed GT row in {path.name}: {line[:60]!r}")
        if fields[0].strip():
            tokens.append(fields[0].strip())
    if not tokens:
        raise ValueError(f"Empty GT: {path.name}")
    return " ".join(tokens)

def normalize(s):
    return " ".join(unicodedata.normalize("NFKC", s).casefold().split())

def metrics(ref, hyp):
    ref, hyp = normalize(ref), normalize(hyp)
    a, b = Counter(ref.split()), Counter(hyp.split())
    hits = sum((a & b).values())
    p = hits / sum(b.values()) if b else 0
    r = hits / sum(a.values())
    return cer(ref, hyp), wer(ref, hyp), p, r, 2*p*r/(p+r) if hits else 0

assert metrics("Ａ b", "a B") == (0, 0, 1, 1, 1)
assert metrics("a", "") == (1, 1, 0, 0, 0)
assert metrics("a a", "a")[2:] == (1, .5, 2/3)
assert metrics("a b", "b a")[1:] == (1, 1, 1, 1)

In [ ]:
%%benchmark
from pathlib import Path
weights = {}
revisions = {}
revisions = {'florence-community/Florence-2-base': '00921df66db728a9ceb750f5eca43e5c203a2051'}
for repo, rev in revisions.items():
    weights[repo] = snapshot_download(repo, revision=rev, ignore_patterns=["*.bin","*.onnx","*.msgpack"])

def trim_generation(sequence, eos_ids, limit):
    # Florence is encoder-decoder: only the first token is decoder_start_token.
    # Trim each sample at its own EOS before decoding, ignoring batch padding.
    generated = sequence[1:]
    stop = next((i + 1 for i, token in enumerate(generated) if token in eos_ids), len(generated))
    return sequence[:stop + 1], stop >= limit


def load_reader(model, weights, image_threads):
    import torch
    from PIL import Image
    from concurrent.futures import ThreadPoolExecutor
    from transformers import AutoProcessor, Florence2ForConditionalGeneration
    folder = next(iter(weights.values()))
    processor = AutoProcessor.from_pretrained(folder)
    engine = Florence2ForConditionalGeneration.from_pretrained(
        folder, dtype=torch.float16, attn_implementation="sdpa").to("cuda").eval()
    text, limit = "<OCR>", 1024
    eos = engine.generation_config.eos_token_id
    if eos is None:
        eos = processor.tokenizer.eos_token_id
    eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos])
    image_pool = ThreadPoolExecutor(max_workers=image_threads)

    def open_image(path):
        with Image.open(path) as im:
            return im.convert("RGB")

    @torch.inference_mode()
    def read(paths):
        images, inputs, ids = [], None, None
        try:
            images = list(image_pool.map(open_image, paths))
            sizes = [image.size for image in images]
            inputs = processor(images=images, text=[text] * len(images),
                               padding=True, return_tensors="pt").to("cuda", torch.float16)
            ids = engine.generate(**inputs, do_sample=False, num_beams=1,
                                  max_new_tokens=limit, use_cache=True)
            sequences = ids.cpu().tolist()
            trimmed = [trim_generation(seq, eos_ids, limit) for seq in sequences]
            answers = processor.batch_decode([seq for seq, _ in trimmed], skip_special_tokens=False)
            return [(processor.post_process_generation(answer, task=text, image_size=size)[text], truncated)
                    for answer, size, (_, truncated) in zip(answers, sizes, trimmed)]
        finally:
            for image in images:
                image.close()
            # Release tensors before an OOM is handled by the caller.
            del inputs, ids

    read.close = lambda: image_pool.shutdown(wait=True)
    return read


weight_files = [p for folder in weights.values() for p in Path(folder).rglob("*")
                if p.suffix in {".safetensors", ".pdiparams", ".pt", ".pth", ".traineddata"}]
assert weight_files, "Inspect weight filenames for this model"
weight_mib = sum(p.stat().st_size for p in weight_files) / 2**20

In [ ]:
%%benchmark
import inspect, platform, threading, time
from concurrent.futures import ThreadPoolExecutor, wait


def evaluate(job):
    import gc, hashlib, json, sqlite3, time
    from pathlib import Path
    import torch
    torch.set_num_threads(job["config"]["threads"])
    torch.cuda.set_device(0)  # CUDA_VISIBLE_DEVICES maps this worker's physical GPU to cuda:0
    device = torch.cuda.get_device_name()
    cfg = dict(job["config"], device=device)
    key = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode()).hexdigest()
    checkpoint = Path(job["output"]) / f"florence-{key[:16]}-{job['worker']}.sqlite"
    db = sqlite3.connect(checkpoint)
    read = None
    label = f"GPU {job['worker']} ({device})"
    started = time.monotonic()
    processed = errors = truncated_count = 0
    batch_size = cfg["batch_size"]
    last_log = 0

    def emit(status):
        print("__OPRB_PROGRESS__" + json.dumps(dict(
            worker=job["worker"], status=status, total=len(job["pages"]),
            resumed=resumed, processed=processed, errors=errors)), flush=True)

    try:
        db.execute("CREATE TABLE IF NOT EXISTS config (json TEXT PRIMARY KEY)")
        db.execute("INSERT OR IGNORE INTO config VALUES (?)", (json.dumps(cfg, sort_keys=True),))
        db.execute("""CREATE TABLE IF NOT EXISTS pages (
            region TEXT, stem TEXT, prediction TEXT, seconds REAL, truncated INTEGER, error TEXT,
            PRIMARY KEY(region, stem))""")
        db.execute("""CREATE TABLE IF NOT EXISTS batch_timings (
            region TEXT, stem TEXT, batch_size INTEGER, batch_seconds REAL,
            PRIMARY KEY(region, stem))""")
        db.commit()
        done = set(db.execute("SELECT region, stem FROM pages WHERE error IS NULL"))
        pending = [(r, s) for r, s in job["pages"] if (r, s) not in done]
        resumed = len(job["pages"]) - len(pending)
        print(f"[{label}] checkpoint={checkpoint.name}; resume={resumed}; pending={len(pending)}", flush=True)
        emit("loading model" if pending else "done")
        if not pending:
            return
        read = load_reader("florence", job["weights"], cfg["image_threads"])

        def paths(keys):
            return [Path(job["root"]) / r / "img" / (s + ".jpg") for r, s in keys]

        # Warm up a full target batch even when the pilot assigns fewer pages to a GPU.
        emit("warm-up")
        while True:
            oom = False
            try:
                keys = (pending * ((batch_size + len(pending) - 1) // len(pending)))[:batch_size]
                read(paths(keys))
                torch.cuda.synchronize()
            except torch.cuda.OutOfMemoryError:
                if batch_size == 1:
                    raise
                oom = True
            except Exception as exc:
                print(f"[{label}] warm-up failed; isolate/retry pages in the main loop: {exc!r}", flush=True)
                break
            if not oom:
                break
            gc.collect()
            torch.cuda.empty_cache()
            batch_size = max(1, batch_size // 2)
            print(f"[{label}] warm-up OOM: reducing batch to {batch_size}", flush=True)
        torch.cuda.reset_peak_memory_stats()
        started = time.monotonic()
        emit("running")

        def run_batch(keys):
            nonlocal processed, errors, truncated_count, batch_size, last_log
            failure = None
            oom = False
            try:
                torch.cuda.synchronize()
                start = time.perf_counter()
                outputs = read(paths(keys))
                torch.cuda.synchronize()
                elapsed = time.perf_counter() - start
                if len(outputs) != len(keys):
                    raise RuntimeError("Reader output count differs from input count")
            except Exception as exc:
                failure = repr(exc)
                oom = isinstance(exc, torch.cuda.OutOfMemoryError)
            # Retry outside the except block so the traceback no longer holds GPU tensors.
            if failure is not None:
                if oom:
                    gc.collect()
                    torch.cuda.empty_cache()
                if len(keys) > 1:
                    middle = len(keys) // 2
                    if oom:
                        batch_size = min(batch_size, middle)
                    print(f"[{label}] split {len(keys)} pages into smaller batches: {failure}", flush=True)
                    run_batch(keys[:middle])
                    run_batch(keys[middle:])
                    return
                r, s = keys[0]
                with db:
                    db.execute("INSERT OR REPLACE INTO pages VALUES (?,?,?,?,?,?)",
                               (r, s, None, None, None, failure))
                    db.execute("DELETE FROM batch_timings WHERE region=? AND stem=?", (r, s))
                errors += 1
                print(f"[{label}] ERROR {r}/{s}: {failure}", flush=True)
            else:
                with db:
                    for (r, s), (prediction, truncated) in zip(keys, outputs):
                        db.execute("INSERT OR REPLACE INTO pages VALUES (?,?,?,?,?,?)",
                                   (r, s, prediction, elapsed / len(keys), int(truncated), None))
                        db.execute("INSERT OR REPLACE INTO batch_timings VALUES (?,?,?,?)",
                                   (r, s, len(keys), elapsed))
                        truncated_count += int(truncated)
            processed += len(keys)
            now = time.monotonic()
            emit("running")
            if last_log == 0 or now - last_log >= job["progress_seconds"] or processed == len(pending):
                speed = processed / max(now - started, 1e-9)
                eta = (len(pending) - processed) / speed
                print(f"[{label}] {resumed + processed}/{len(job['pages'])} | "
                      f"{speed:.2f} pages/s | ETA {eta/60:.1f} min | "
                      f"batch={len(keys)} (next <= {batch_size}) | errors={errors} | "
                      f"truncated={truncated_count} | peak VRAM allocated/reserved="
                      f"{torch.cuda.max_memory_allocated()/2**30:.2f}/"
                      f"{torch.cuda.max_memory_reserved()/2**30:.2f} GiB", flush=True)
                last_log = now

        offset = 0
        while offset < len(pending):
            # Do not mix text and nontext pages in a timed batch.
            end = offset + 1
            while end < min(offset + batch_size, len(pending)) and pending[end][0] == pending[offset][0]:
                end += 1
            run_batch(pending[offset:end])
            offset = end
        emit("done")
    finally:
        if read is not None:
            read.close()
        db.close()


worker_code = ("import sys, json\n" + inspect.getsource(trim_generation) + "\n" +
               inspect.getsource(load_reader) + "\n" + inspect.getsource(evaluate) +
               "\nevaluate(json.load(sys.stdin))\n")
from importlib.metadata import distributions
versions = {d.metadata["Name"].lower(): d.version for d in distributions() if d.metadata["Name"]}
workers = GPU_WORKERS
config = dict(model=MODEL, dataset=DATA_REV, revisions=revisions, weights_mib=weight_mib,
              prompt="<OCR>", workers=workers, versions=versions, cpu=platform.machine(),
              threads=CPU_THREADS, image_threads=IMAGE_THREADS, batch_size=BATCH_SIZE,
              timing_mode="single_page" if BATCH_SIZE == 1 else "batch_amortized",
              precision="float16", attention="sdpa", max_new_tokens=1024,
              do_sample=False, num_beams=1, use_cache=True,
              reader_sha256=hashlib.sha256(worker_code.encode()).hexdigest())
if Path("/proc/cpuinfo").exists():
    config["cpu"] = next((line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                          if line.startswith("model name")), config["cpu"])
positions = {page: i for i, page in enumerate(pages)}
states = {i: dict(total=sum(positions[p] % workers == i for p in selected),
                  resumed=0, processed=0, errors=0, status="starting") for i in range(workers)}
state_lock = threading.Lock()


def relay(index, process):
    # Explicit pipes -> private kernel -> notebook: avoids buffered/invisible subprocess logs.
    for line in process.stdout:
        if line.startswith("__OPRB_PROGRESS__"):
            state = json.loads(line[len("__OPRB_PROGRESS__"):])
            with state_lock:
                states[index] = state
        else:
            print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"GPU {index} worker exited with code {process.returncode}; see its log above")


def show_progress(start):
    with state_lock:
        snapshot = [dict(s) for s in states.values()]
    total = sum(s["total"] for s in snapshot)
    completed = sum(s["resumed"] + s["processed"] for s in snapshot)
    new = sum(s["processed"] for s in snapshot)
    errors = sum(s["errors"] for s in snapshot)
    elapsed = time.monotonic() - start
    speed = new / max(elapsed, 1e-9)  # Includes startup, warm-up and writes for wall-time ETA.
    eta = f"{(total-completed)/speed/60:.1f} min" if speed else "waiting for first batch"
    print(f"[TOTAL] {completed}/{total} ({100*completed/max(total,1):.1f}%) | "
          f"errors={errors} | {speed:.2f} pages/s | elapsed={elapsed/60:.1f} min | ETA {eta} | "
          + ", ".join(f"GPU {i}: {s['status']}" for i, s in enumerate(snapshot)), flush=True)


processes = []
pool = ThreadPoolExecutor(max_workers=workers)
start = time.monotonic()
try:
    futures = []
    for index in range(workers):
        job = dict(pages=[p for p in selected if positions[p] % workers == index], config=config,
                   output=str(OUTPUT), root=str(ROOT), weights=weights, worker=index,
                   progress_seconds=PROGRESS_SECONDS)
        env = dict(os.environ, USE_TF="0", USE_FLAX="0", CUDA_VISIBLE_DEVICES=str(index),
                   OMP_NUM_THREADS=str(CPU_THREADS), OMP_THREAD_LIMIT=str(CPU_THREADS),
                   TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
        proc = subprocess.Popen([str(model_python), "-I", "-u", "-c", worker_code],
                                stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding="utf-8", env=env, bufsize=1)
        processes.append(proc)
        futures.append(pool.submit(relay, index, proc))
        proc.stdin.write(json.dumps(job))
        proc.stdin.close()
    remaining = set(futures)
    while remaining:
        finished, remaining = wait(remaining, timeout=PROGRESS_SECONDS, return_when="FIRST_COMPLETED")
        for future in finished:
            future.result()
        show_progress(start)
finally:
    # Also stop child GPU processes on notebook interruption or a worker failure.
    for proc in processes:
        if proc.poll() is None:
            proc.terminate()
    for proc in processes:
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
    pool.shutdown(wait=True)

### Score saved predictions

Tables report page-level macro means and separate configuration hashes. Check successful and expected counts before reporting full results. `timing_mode=batch_amortized` means **batch seconds / actual batch size**, not latency of one page; use BATCH_SIZE=1 for single-page latency comparisons. Different batch sizes, SDPA and the optimized reader create independent checkpoints. Old pilots are displayed separately and cannot be treated as coverage for a new config. The region-based wall-time projection is approximate and assumes balanced workers; the TOTAL log measures this session's actual wall throughput.

In [ ]:
%%benchmark
from contextlib import closing
def occlusion_metadata(stem):
    import re
    parts = stem.split("__", 2)
    density = parts[1] if len(parts) > 1 and re.fullmatch(r"[0-9]+(?:p[0-9]+)?pct", parts[1]) else "unknown"
    return parts[0], density

def score_results():
    rows = []
    refs = {}
    configs = []
    for path in sorted(OUTPUT.glob(f"{MODEL}-*.sqlite")):
        with closing(sqlite3.connect(path)) as db:
            cfg, = db.execute("SELECT json FROM config").fetchone()
            config = json.loads(cfg)
            if config["dataset"] != DATA_REV:
                continue
            configs.append(dict(config=hashlib.sha256(cfg.encode()).hexdigest()[:12],
                                batch_size=config.get("batch_size", 1),
                                timing_mode=config.get("timing_mode", "single_page"),
                                attention=config.get("attention", "auto")))
            for region, stem, pred, seconds, trunc, error in db.execute("SELECT * FROM pages"):
                key = (region, stem)
                if key not in refs:
                    refs[key] = reference(*key)
                scores = (None,)*5 if error else metrics(refs[key], pred)
                parts = occlusion_metadata(stem)
                rows.append((config["model"], hashlib.sha256(cfg.encode()).hexdigest()[:12], config["device"],
                             config["workers"], config["weights_mib"], region, stem, parts[0], parts[1],
                             pred, refs[key], seconds, trunc, error, *scores))
    cols = ["model","config","device","workers","weights_mib","region","stem","type","density",
            "prediction","gt","seconds","truncated","error","cer","wer","precision","recall","f1"]
    print("Timing protocols (do not compare amortized throughput with single-page latency):")
    display(pd.DataFrame(configs).drop_duplicates())
    return pd.DataFrame(rows, columns=cols)

results = score_results()
assert len(results) and not results.duplicated(["model","config","region","stem"]).any()
keys = ["model","config"]
values = ["cer","wer","precision","recall","f1","seconds"]
coverage = results.groupby(keys + ["region"]).agg(attempted=("stem","size"), ok=("seconds","count"),
                                                   truncated=("truncated","sum"))
coverage["expected"] = coverage.index.get_level_values("region").map(COUNTS)
display(coverage)
ok = results[results.error.isna()]
display(ok.groupby(keys+["device","weights_mib"])[values].mean())
for groups in (["region"], ["region","type"], ["region","density"]):
    display(ok.groupby(keys+groups)[values].mean().join(ok.groupby(keys+groups).size().rename("pages")))
latency = ok.groupby(keys+["workers","region"]).seconds.mean().unstack().reindex(columns=list(COUNTS))
display((latency.mul(pd.Series(COUNTS)).sum(axis=1, skipna=False) /
         (3600 * latency.index.get_level_values("workers"))).rename("projected_wall_hours"))
display(pd.DataFrame([(r, *occlusion_metadata(s)) for r,s in pages],
                     columns=["region","type","density"]).value_counts().rename("dataset_pages"))
# Locate qualitative candidates. Inspect the same image crop, GT and predictions before using them.
for region, group in ok.groupby("region"):
    print(region)
    display(pd.concat([group.nsmallest(1,"wer"),group.nlargest(1,"wer")])[
        ["model","stem","wer","gt","prediction"]])
# Example: display(Image.open(ROOT / region / "img" / (stem + ".jpg")))